In [1]:
import os
from hwcomponents_adc import ADC
import functools

In [2]:
@functools.lru_cache(maxsize=None)
def get_adc_dac_area_energy_scaling(
    resolution_original: int,
    resolution_new: int,
    technology_original: float,
    technology_new: float,
    throughput_original: float,
    throughput_new: float,
) -> tuple[float, float]:
    """
    Returns a tuple of [area scale, energy scale]

    resolution_original: original resolution of the ADC and DAC (bits)
    resolution_new: new resolution of the ADC and DAC (bits)
    technology_original: original technology of the ADC and DAC (meters)
    technology_new: new technology of the ADC and DAC (meters)
    throughput_original: original throughput of the ADC and DAC (Hz)
    throughput_new: new throughput of the ADC and DAC (Hz)
    """

    print(f'Scaling from n_bits {resolution_original} -> {resolution_new}, tech node {technology_original} -> {technology_new}, throughput {throughput_original} -> {throughput_new}')

    adc_original = ADC(
        n_bits=resolution_original,
        tech_node=technology_original,
        throughput=throughput_original,
    )

    adc_new = ADC(
        n_bits=resolution_new,
        tech_node=technology_new,
        throughput=throughput_new,
    )

    return adc_new.area / adc_original.area, adc_new.read() / adc_original.read()

@functools.lru_cache(maxsize=None)
def make_cell_file(size_scale: int) -> str:
    cell_dir = os.path.join(os.getcwd(), "cell_files")
    os.makedirs(cell_dir, exist_ok=True)
    cell_file = os.path.join(cell_dir, f"cell_{size_scale}.yaml")
    if os.path.exists(cell_file):
        return cell_file
    content = open(os.path.join(os.getcwd(), "sram.cell.yaml")).read()
    area_line = [l for l in content.split("\n") if "-CellArea" in l][0]
    area_line_number = int(area_line.split(":")[1].strip())
    new_area = area_line_number * size_scale
    content = content.replace(area_line, f"-CellArea (F^2): {new_area}")
    access_cmos_line = [l for l in content.split("\n") if "-AccessCMOSWidth" in l][0]
    access_cmos_line_number = int(access_cmos_line.split(":")[1].strip())
    content = content.replace(access_cmos_line, f"-AccessCMOSWidth (F): {access_cmos_line_number * size_scale}")
    with open(cell_file, "w") as f:
        f.write(content)
    return cell_file


In [3]:
from _tests import scripts, THIS_SCRIPT_DIR, utl, MACRO_NAME
import functools
import joblib
import copy

ARRAY_THROUGHPUT = 1e7
COLUMNS_PER_ADC = 64
COLUMNS_PER_DAC = 64
TECHNOLOGY = 32e-9

cache2 = joblib.Memory(location="/tmp/deleteme")

@functools.lru_cache(maxsize=None)
# @cache2.cache
def run_full_system_dnn_no_scale(
    dnn_name: str,
    input_bits: int,
    output_bits: int,
    weight_bits: int,
    array_size: int,
    adc_bits: int,
):
    dnn_dir = utl.path_from_model_dir(f"workloads/{dnn_name}")
    layer_paths = [
        os.path.join(dnn_dir, l) for l in os.listdir(dnn_dir) if l.endswith(".yaml")
    ]

    layer_paths = [l for l in layer_paths if "From einsum" not in open(l, "r").read()]

    if "gpt2_medium" in dnn_name:
        layer_paths = layer_paths[:-1]

    def callfunc(spec):
        spec.architecture.find("row").spatial.meshY = array_size
        spec.architecture.find("column").spatial.meshX = array_size

    sram_file = make_cell_file(1)

    results = utl.parallel_test(
        utl.delayed(utl.run_layer)(
            macro=MACRO_NAME,
            layer=l,
            variables=dict(
                INPUT_BITS=input_bits,
                OUTPUT_BITS=output_bits,
                WEIGHT_BITS=weight_bits,
                ADC_RESOLUTION=adc_bits,
                VOLTAGE_DAC_RESOLUTION=input_bits,
                SUPPORTED_INPUT_BITS=input_bits,
                SUPPORTED_OUTPUT_BITS=output_bits,
                SUPPORTED_WEIGHT_BITS=weight_bits,
                BITS_PER_CELL=1,
                TECHNOLOGY=round(TECHNOLOGY * 1e9),
                CELL_CONFIG=f'"{sram_file}"',
                BASE_LATENCY=1/ARRAY_THROUGHPUT,
                ADC_ENERGY_SCALE=1,
                ADC_AREA_SCALE=1,
                DAC_ENERGY_SCALE=1,
                DAC_AREA_SCALE=1,
            ),
            callfunc=callfunc,
        )
        for l in layer_paths
    )
    results.clear_zero_energies()
    results.clear_zero_areas()
    return results.aggregate()


def run_full_system_dnn_scale(
    dnn_name: str,
    input_bits: int,
    output_bits: int,
    weight_bits: int,
    array_size: int,
    adc_bits: int,
    sram_size_scale: int,

    reference_adc_resolution: int,
    reference_adc_tech_node: float,
    reference_adc_throughput: float,
    reference_adc_energy: float,
    reference_adc_area: float,

    reference_dac_resolution: int,
    reference_dac_tech_node: float,
    reference_dac_throughput: float,
    reference_dac_energy: float,
    reference_dac_area: float,
):
    results = run_full_system_dnn_no_scale(
        dnn_name,
        input_bits,
        output_bits,
        weight_bits,
        array_size,
        adc_bits,
    )

    # Run array at 10MHz, split an ADC between every 32 columns
    adc_throughput = ARRAY_THROUGHPUT * COLUMNS_PER_ADC
    dac_throughput = ARRAY_THROUGHPUT * COLUMNS_PER_DAC
    n_adcs = array_size // COLUMNS_PER_ADC
    n_dacs = array_size // COLUMNS_PER_DAC

    adc_energy_scale, adc_area_scale = get_adc_dac_area_energy_scaling(
        resolution_original=reference_adc_resolution,
        resolution_new=adc_bits,
        technology_original=reference_adc_tech_node * 1e-6,
        technology_new=32e-9,
        throughput_original=reference_adc_throughput,
        throughput_new=adc_throughput,
    )
    dac_energy_scale, dac_area_scale = get_adc_dac_area_energy_scaling(
        resolution_original=reference_dac_resolution,
        resolution_new=output_bits,
        technology_original=reference_dac_tech_node * 1e-6,
        technology_new=32e-9,
        throughput_original=reference_dac_throughput,
        throughput_new=dac_throughput,
    )

    def adjust_energy(component_name: str, energy_scale: float):
        orig = results.per_component_energy[component_name]
        results.per_component_energy[component_name] = orig * energy_scale
        results.energy += orig * (energy_scale - 1)

    def adjust_area(component_name: str, area_scale: float):
        orig = results.per_component_area[component_name]
        results.per_component_area[component_name] = orig * area_scale
        results.area += orig * (area_scale - 1)

    results = copy.copy(results)
    results.per_component_energy = copy.copy(results.per_component_energy)
    results.per_component_area = copy.copy(results.per_component_area)

    results.per_component_energy["adc"] *= reference_adc_energy * adc_energy_scale
    results.per_component_area["adc"] *= reference_adc_area * adc_area_scale * n_adcs
    results.per_component_energy["dac"] *= reference_dac_energy * dac_energy_scale
    results.per_component_area["dac"] *= reference_dac_area * dac_area_scale * n_dacs
    results.per_component_energy["cim_unit"] *= sram_size_scale
    results.per_component_area["cim_unit"] *= sram_size_scale
    # results.per_component_energy["row_drivers"] *= sram_size_scale
    print(f'Energy: {results.per_component_energy}')
    print(f'Area: {results.per_component_area}')
    results.energy = sum(results.per_component_energy.values())
    results.area = sum(results.per_component_area.values())
    return results

In [4]:
import pandas as pd

xlsx = "./Case_study_params.xlsx"

# Read three sheets:
# - ADC. Columns are TECHNOLOGY, Area [mm^2], fs [Hz], P/fs [J]
# - DAC. Columns are Technology, Area (mm^2), Fs MS, P/fs
# - Memory-cell. Columns are Technology, Size

adc = pd.read_excel(xlsx, sheet_name="ADC")
dac = pd.read_excel(xlsx, sheet_name="DAC")
memory_cell = pd.read_excel(xlsx, sheet_name="Memory-cell")

adc_renames = {
    "bits": ("reference_adc_resolution", 1),
    "TECHNOLOGY": ("reference_adc_tech_node", 1),
    "AREA [mm^2]": ("reference_adc_area", 1e-6),
    "fs [Hz]": ("reference_adc_throughput", 1),
    "P/fs [J]": ("reference_adc_energy", 1),
    "stdev noise": ("adc_stdev_noise", 1),
}
dac_renames = {
    "M": ("reference_dac_resolution", 1),
    "Technology": ("reference_dac_tech_node", 1),
    "Area (mm2)": ("reference_dac_area", 1e-6),
    "Fs MS": ("reference_dac_throughput", 1e-6),
    "P/fs": ("reference_dac_energy", 1),
    "stdev noise compared to V_fsr (full scale range)": ("dac_stdev_noise", 1),
}
memory_cell_renames = {
    "Technology": ("reference_memory_cell_tech_node", 1),
    "Size": ("reference_memory_cell_size", 1),
    "Stdev Error (to FSR)": ("memory_cell_stdev_error", 1),
}

def agg_rename(df: pd.DataFrame, renames: dict[str, tuple[str, float]]) -> pd.DataFrame:
    column_renames, column_scales = zip(*renames.values())
    df = df[list(renames.keys())].rename(columns={k: v for k, v in zip(renames.keys(), column_renames)})
    for col in column_renames:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna()

    for col, scale in zip(column_renames, column_scales):
        df[col] *= scale
    return df

aggregated_adc = agg_rename(adc, adc_renames)
aggregated_dac = agg_rename(dac, dac_renames)
aggregated_memory_cell = agg_rename(memory_cell, memory_cell_renames)

In [5]:
results = {}
adc_rows = list(aggregated_adc.itertuples())
import tqdm

for adc_row in tqdm.tqdm(adc_rows):
    for dac_row in aggregated_dac.itertuples():
        for memory_cell_row in aggregated_memory_cell.itertuples():
            result = run_full_system_dnn_scale(
                "resnet50",
                input_bits=8,
                output_bits=8,
                weight_bits=8,
                array_size=1152,
                adc_bits=12,
                sram_size_scale=memory_cell_row.reference_memory_cell_size,
                reference_adc_resolution=adc_row.reference_adc_resolution,
                reference_adc_tech_node=adc_row.reference_adc_tech_node,
                reference_adc_throughput=adc_row.reference_adc_throughput,
                reference_adc_energy=adc_row.reference_adc_energy,
                reference_adc_area=adc_row.reference_adc_area,
                reference_dac_resolution=dac_row.reference_dac_resolution,
                reference_dac_tech_node=dac_row.reference_dac_tech_node,
                reference_dac_throughput=dac_row.reference_dac_throughput,
                reference_dac_energy=dac_row.reference_dac_energy,
                reference_dac_area=dac_row.reference_dac_area,
            )
            energy = result.energy
            area = result.area
            print(f'Energy: {energy}, Area: {area}')
            results[adc_row.adc_stdev_noise, dac_row.dac_stdev_noise, memory_cell_row.memory_cell_stdev_error] = energy * area

print(results)

 39%|███▉      | 19/49 [00:08<00:09,  3.07it/s]

Scaling from n_bits 12 -> 12, tech node 5.4999999999999996e-08 -> 3.2e-08, throughput 2048000 -> 640000000.0
Scaling from n_bits 8.0 -> 8, tech node 1.4e-08 -> 3.2e-08, throughput 0.055999999999999994 -> 640000000.0
Energy: {'cim_unit': 9.28805552128e-08, 'row_drivers': 6.50198931996672e-07, 'dac': 0.028031914120446696, 'adc': 0.030400295142805463}
Area: {'adc': 0.21543174031030513, 'dac': 0.03313832598765739, 'row_drivers': 2.4642969600000003e-05, 'cim_unit': 0.00089060441849856}
Energy: 0.05843295234273937, Area: 0.24948531368606108
Energy: {'cim_unit': 1.857611104256e-07, 'row_drivers': 6.50198931996672e-07, 'dac': 0.028031914120446696, 'adc': 0.030400295142805463}
Area: {'adc': 0.21543174031030513, 'dac': 0.03313832598765739, 'row_drivers': 2.4642969600000003e-05, 'cim_unit': 0.00178120883699712}
Energy: 0.05843304522329458, Area: 0.2503759181045596
Energy: {'cim_unit': 3.715222208512e-07, 'row_drivers': 6.50198931996672e-07, 'dac': 0.028031914120446696, 'adc': 0.030400295142805463

100%|██████████| 49/49 [00:08<00:00,  5.54it/s]

Energy: {'cim_unit': 9.28805552128e-08, 'row_drivers': 6.50198931996672e-07, 'dac': 0.028031914120446696, 'adc': 0.9703334489414388}
Area: {'adc': 2.3567291502749796, 'dac': 0.03313832598765739, 'row_drivers': 2.4642969600000003e-05, 'cim_unit': 0.00089060441849856}
Energy: 0.9983661061413727, Area: 2.3907827236507355
Energy: {'cim_unit': 1.857611104256e-07, 'row_drivers': 6.50198931996672e-07, 'dac': 0.028031914120446696, 'adc': 0.9703334489414388}
Area: {'adc': 2.3567291502749796, 'dac': 0.03313832598765739, 'row_drivers': 2.4642969600000003e-05, 'cim_unit': 0.00178120883699712}
Energy: 0.9983661990219279, Area: 2.391673328069234
Energy: {'cim_unit': 3.715222208512e-07, 'row_drivers': 6.50198931996672e-07, 'dac': 0.028031914120446696, 'adc': 0.9703334489414388}
Area: {'adc': 2.3567291502749796, 'dac': 0.03313832598765739, 'row_drivers': 2.4642969600000003e-05, 'cim_unit': 0.00356241767399424}
Energy: 0.9983663847830383, Area: 2.3934545369062312
Energy: {'cim_unit': 7.430444417024e-07

In [ ]:
import matplotlib.pyplot as plt

x_index = 2
x_labels = [
    "ADC noise",
    "DAC noise",
    "Memory cell noise",
]
x_label = x_labels[x_index]

x = [r[x_index] for r in results.keys()]
y = list(results.values())
plt.scatter(x, y)
plt.xscale("log")
plt.yscale("log")
plt.xlabel(x_label)
plt.ylabel("Energy * Area")
plt.show()

print(results)

import plotly
x_index = 0
y_index = 1
color_index = 2
x_label = x_labels[x_index]
y_label = x_labels[y_index]
color_label = x_labels[color_index]
z_label = "Energy * Area"
x = [r[x_index] for r in results.keys()]
y = [r[y_index] for r in results.keys()]
z = list(results.values())
c = [r[color_index] for r in results.keys()]

# Make a 3D plot of energy vs area vs noise
import plotly.graph_objects as go

# Make it logscale
fig = go.Figure(data=[go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(size=10, color=c, colorscale='Viridis'))])
fig.update_layout(scene=dict(xaxis=dict(type='log'), yaxis=dict(type='log'), zaxis=dict(type='log')))
fig.update_traces(marker=dict(size=10, color=c, colorscale='Viridis'))
fig.update_layout(scene=dict(xaxis=dict(title=x_label), yaxis=dict(title=y_label), zaxis=dict(title=z_label)))
fig.show()